# Lesson 02 - Verkennen van het Microsoft Agent Framework

Het **Microsoft Agent Framework (MAF)** is een uniform framework voor het bouwen van AI-agenten. Het biedt een duidelijke, samenstelbare architectuur met vier kerncomponenten:

- **Client** – maakt verbinding met een AI-modelendpoint en beheert de communicatie
- **Agent** – wikkelt een client in met instructies en tooldefinities
- **Tools** – breiden de mogelijkheden van de agent uit met aangepaste functies die het model kan aanroepen
- **Session** – onderhoudt de gespreksgeschiedenis voor meervoudige interacties

In deze les bouwen we een **reisboeking agent** die de beschikbaarheid van bestemmingen controleert met behulp van deze concepten.


## Installatie


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Begrijpen van de Agent Framework Architectuur

Het Microsoft Agent Framework volgt een gelaagde architectuur:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – Een `AzureAIProjectAgentProvider` maakt verbinding met een Azure OpenAI-implementatie. Het regelt authenticatie, aanvraagformaat en antwoordparsing.
2. **Agent** – Gemaakt vanuit de client via `provider.create_agent()`, combineert de agent modeltoegang met instructies (systeem prompt) en tools.
3. **Tools** – Python-functies gedecoreerd met `@tool` die de agent kan aanroepen om acties uit te voeren of gegevens op te halen.
4. **Sessie** – Een `AgentSession` object (gemaakt via `agent.create_session()`) dat de gespreksgeschiedenis opslaat, waardoor multi-turn dialoog mogelijk is waarbij de agent zich eerdere context herinnert.

Laten we elke laag stap voor stap opbouwen.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Tools toevoegen met de @tool Decorator

Tools laten agents acties uitvoeren die verder gaan dan het genereren van tekst. De `@tool` decorator zet een gewone Python-functie om in iets dat de agent kan aanroepen.

Belangrijke punten:
- Gebruik `Annotated[type, "description"]` zodat het model elk parameter begrijpt.
- De docstring wordt de toolbeschrijving die het model ziet.
- `approval_mode="never_require"` betekent dat de tool automatisch wordt uitgevoerd zonder gebruikersbevestiging.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Een Agent maken met Hulpmiddelen

Nu combineren we de client, instructies en hulpmiddelen in een agent. De `instructions` fungeren als de systeemprompt — ze definiëren de persoonlijkheid en het gedrag van de agent.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Meerdere beurten gesprekken met sessies

Een `AgentSession` (gemaakt via `agent.create_session()`) houdt alle berichten in een gesprek bij. Door dezelfde sessie door te geven aan elke `agent.run()`-oproep, heeft de agent toegang tot de volledige gesprekshistorie en kan hij terugverwijzen naar eerdere berichten.

We geven `tools=[check_destination_availability]` door zodat de agent onze beschikbaarheidschecker tijdens elke beurt kan aanroepen.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Samenvatting

In deze les heb je de vier pijlers van het Microsoft Agent Framework onderzocht:

| Concept | Wat je hebt geleerd |
|---------|---------------------|
| **Client** | `AzureAIProjectAgentProvider` maakt verbinding met Azure OpenAI met op referenties gebaseerde authenticatie |
| **Agent** | `provider.create_agent()` bundelt een modelverbinding met instructies en een naam |
| **Tools** | De `@tool` decorator maakt Python-functies beschikbaar voor de agent om aan te roepen |
| **Session** | `agent.create_session()` onderhoudt de gespreksgeschiedenis over meerdere rondes |

Deze bouwstenen vormen samen agents die natuurlijke gesprekken kunnen voeren, externe functies kunnen aanroepen en context kunnen behouden — de basis voor meer geavanceerde agentpatronen in latere lessen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, kan het zijn dat automatische vertalingen fouten of onnauwkeurigheden bevatten. Het originele document in de oorspronkelijke taal dient als de gezaghebbende bron te worden beschouwd. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
